In [1]:
import os
import time
import psutil
from concurrent.futures import ThreadPoolExecutor
from llama_cpp import Llama
import gc

In [ ]:
MODEL_PATH = os.path.join(os.getcwd(), "Qwen3.5-4B-Q6_K.gguf")
RAM_PER_INSTANCE_MB = 3600  # Conservative estimate
MAX_CHUNK_TOKENS = 1600
CONTEXT_SIZE = 2048
file_bytes = os.path.getsize(MODEL_PATH)
size_mb = file_bytes / (1024 * 1024)
size_gb = file_bytes / (1024 * 1024 * 1024)
print(f"Model size {size_mb:.2f} MB or {size_gb:.2f} GB")

Model size 1484.29 MB or 1.45 GB


In [5]:
tokenizer_llm = Llama(model_path=MODEL_PATH, vocab_only=True, verbose=False)

llama_context: n_ctx_seq (512) > n_ctx_train (0) -- possible training context overflow


In [13]:
def get_token_count(text):
    return len(tokenizer_llm.tokenize(text.encode("utf-8")))

def chunk_diff(diff_text, max_tokens):
    tokens = tokenizer_llm.tokenize(diff_text.encode("utf-8"))
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i : i + max_tokens]
        chunks.append(tokenizer_llm.detokenize(chunk_tokens).decode("utf-8"))
    return chunks

def process_chunk(chunk_id, chunk_text):
    """Worker function to summarize a single chunk"""
    # Create a local instance for this thread to avoid race conditions
    local_llm = Llama(model_path=MODEL_PATH, n_ctx=CONTEXT_SIZE, verbose=False)
    
    prompt = f"""<|im_start|>system
Summarize the code changes in this snippet briefly. Focus on 'what' changed. Keep it under 200 tokens<|im_end|>
<|im_start|>user
{chunk_text}<|im_end|>
<|im_start|>assistant
"""
    output = local_llm(prompt, max_tokens=200, stop=["<|im_end|>"], temperature=0.1)
    summary = output["choices"][0]["text"].strip()
    del local_llm
    print(f"[OK] Chunk {chunk_id} summarized.", flush=True)
    return summary

def generate_final_message(summaries):
    """The 'Reduce' step: Synthesis of all summaries into one commit"""
    final_llm = Llama(model_path=MODEL_PATH, n_ctx=CONTEXT_SIZE, verbose=False)
    combined_summaries = "\n".join([f"- {s}" for s in summaries])
    
    prompt = f"""<|im_start|>system
You are a git commit generator. There was a big git differences 
which was divided into chunks and analyzed independently and generated sumaries for those
chunks. Use the provided summaries to write a single final.
Conventional Commit (feat:, fix:, chore:, etc.). Output ONLY the message.<|im_end|>
<|im_start|>user
{combined_summaries}<|im_end|>
<|im_start|>assistant
feat:"""

    output = final_llm(prompt, max_tokens=150, stop=["<|im_end|>"], temperature=0.2)
    return "feat:" + output["choices"][0]["text"].strip()

In [7]:
with open("new_diff.txt", "r") as f:
        git_diff = f.read()

total_tokens = get_token_count(git_diff)
print(f"Total Diff Tokens: {total_tokens}", flush=True)
chunks = chunk_diff(git_diff, MAX_CHUNK_TOKENS)
print(f"Split into {len(chunks)} chunks.", flush=True)

Total Diff Tokens: 2375


Split into 2 chunks.


In [8]:
available_ram_mb = psutil.virtual_memory().available / (1024 * 1024)
max_workers = int((available_ram_mb - 1000) // RAM_PER_INSTANCE_MB)
max_workers = max(1, min(max_workers, len(chunks))) # At least 1, max = num of chunks

print(f"Available RAM: {available_ram_mb:.0f}MB | Spawning {max_workers} parallel workers...", flush=True)

Available RAM: 3516MB | Spawning 1 parallel workers...


In [9]:
print(f"Starting sequential processing of {len(chunks)} chunks...", flush=True)
start_time = time.time()

sequential_summaries = []

for i, chunk in enumerate(chunks):
    chunk_start = time.time()
    
    # Process the chunk one by one
    summary = process_chunk(i, chunk)
    sequential_summaries.append(summary)
    
    # Optional: Force garbage collection after each chunk to keep RAM low
    gc.collect()
    
    chunk_end = time.time()
    print(f"  -> Chunk {i} took {chunk_end - chunk_start:.2f} seconds.", flush=True)

end_time = time.time()
total_sequential_time = end_time - start_time

print(f"\n--- SEQUENTIAL STATS ---")
print(f"Total time: {total_sequential_time:.2f} seconds")
print(f"Average time per chunk: {total_sequential_time / len(chunks):.2f} seconds")
print(f"------------------------\n")

Starting sequential processing of 2 chunks...


llama_context: n_ctx_seq (2048) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


[OK] Chunk 0 summarized.
  -> Chunk 0 took 19.49 seconds.


llama_context: n_ctx_seq (2048) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


[OK] Chunk 1 summarized.
  -> Chunk 1 took 13.24 seconds.

--- SEQUENTIAL STATS ---
Total time: 32.73 seconds
Average time per chunk: 16.37 seconds
------------------------



In [8]:
# summaries = []
# with ThreadPoolExecutor(max_workers=max_workers) as executor:
#     futures = [executor.submit(process_chunk, i, chunk) for i, chunk in enumerate(chunks)]
#     for future in futures:
#         summaries.append(future.result())
        
# print("Waiting for workers to release resources...", flush=True)
# gc.collect()
# time.sleep(1) 

In [10]:
print(sequential_summaries)

['<think>\nLet me analyze the git diff to understand what changed:\n\n1. **README.md changes:**\n   - Model status updated from "Testing" to "Working" for Qwen2.5 and SmolLM2\n   - New models added: Qwen3.5-0.8B-Q4_K_M, Qwen3-0.6B-Q4_K_S, SmolLM2-135M-Q8\n   - Llama 3.2 (1B) moved to "Roadmap" status\n\n2. **python-test/.gitignore changes:**\n   - Added `test_diff.txt` to gitignore (was removed)\n\n3. **python-test/notebook.ipynb changes:**\n   - Multiple execution counts updated (null → 1, 66, 67, 5, 56, 62)\n   - Model path changed from Qwen2.5 to Qwen', '<think>\nLet me analyze the git diff changes:\n\n1. **Main Python file changes:**\n   - Added a new function `generate_commit_message()` that:\n     - Takes a git diff as input\n     - Uses an LLM to generate a conventional commit message\n     - Uses `stop=["\\n"]` to prevent model from continuing after generating the message\n     - Uses `temperature=0.1` for consistent, logical output\n     - Returns the generated message\n\n2. *

In [14]:
print("\nSynthesizing final commit message...", flush=True)
final_commit = generate_final_message(sequential_summaries)

print("\n--- GENERATED COMMIT MESSAGE ---", flush=True)
print(final_commit, flush=True)


Synthesizing final commit message...


llama_context: n_ctx_seq (2048) < n_ctx_train (262144) -- the full capacity of the model will not be utilized



--- GENERATED COMMIT MESSAGE ---
feat:Added function to generate conventional commit messages from git diffs
